# Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Standard library imports
import bisect
import csv
import hashlib
import io
import math
import re
from collections import Counter
from pathlib import Path

# Third-party imports
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from textblob import TextBlob
from wordcloud import WordCloud, STOPWORDS


#################
# CONFIGURATION #
#################

# Set paths
base_path   = Path(r'/Users/amb/Downloads/Corpora/GTA') ######### ANON #########
export_path = Path.cwd()

# Set prefix for exported files
export_prefix = 'game_studies_gta_'

# Set encodings to try when reading source files
source_encodings = ['utf-8-sig', 'cp1252', 'latin-1']

# Set window size for standardised type-token ratio
sttr_window = 1000

# Set global display settings for pandas
pd.options.display.float_format = '{:,.4f}'.format


##############
# TOKENIZERS #
##############

# Tokenize alphabetic words for linguistic metrics
word_pattern = re.compile(r'\b[a-zA-Z-]{2,}\b')

# Tokenize words and numbers for barcode plots
sequence_pattern = re.compile(r'\b[a-zA-Z0-9-]{2,}\b')


#########################
# CONSPIRACY CATEGORIES #
#########################

# Map keywords to categories
conspiracy_categories = {
    "Aliens and UFOs": [
        "abduct*",
        "alien*",
        "anunnaki",
        "area 51",
        "area 53",
        "area 69",
        "extraterrestrial*",
        "mars",
        "reptilian",
        "saucer*",
        "spaceship",
        "ufo*"
    ],
    "Mind control": [
        "aluminium",
        "aluminum",
        "brainwash*",
        "frequency",
        "microwave",
        "mind control",
        "propaganda",
        "subliminal*",
        "tinfoil"
    ],
    "Deep state": [
        "black ops",
        "bureau",
        "clandestine",
        "deep state",
        "fbi",
        "fib",
        "grey government",
        "iaa"
    ],
    "Surveillance": [
        "echelon",
        "government satellites",
        "nsa",
        "spied",
        "spies",
        "spy",
        "spying",
        "sub-dermal neurophone",
        "surveil*",
        "wiretap*"
    ],
    "Varia": [
        "all seeing eye",
        "clon*",
        "cover-up",
        "elvis",
        "hitler",
        "illuminati",
        "jfk",
        "mason*",
        "new world order",
        "vaccin*"
    ],
    "General conspiracy language": [
        "between the lines",
        "classified",
        "conspiracy",
        "decept*",
        "distort*",
        "hoax*",
        "manipulate*",
        "red pil*",
        "staged",
        "top-secret",
        "unveil*"
    ]
}


#########
# GAMES #
#########

# Define order
ordered_games = [
    "Grand Theft Auto (1997)",
    "Grand Theft Auto: London 1969 (1999)",
    "Grand Theft Auto: London 1961 (1999)",
    "Grand Theft Auto 2 (1999)",
    "Grand Theft Auto III (2001)",
    "Grand Theft Auto: Vice City (2002)",
    "Grand Theft Auto: San Andreas (2004)",
    "Grand Theft Auto Advance (2004)",
    "Grand Theft Auto: Liberty City Stories (2005)",
    "Grand Theft Auto: Vice City Stories (2006)",
    "Grand Theft Auto IV (2008)",
    "Grand Theft Auto IV: The Lost and Damned (2009)",
    "Grand Theft Auto IV: The Ballad of Gay Tony (2009)",
    "Grand Theft Auto: Chinatown Wars (2009)",
    "Grand Theft Auto V and Online (2013 ff.)"
]

# Map abbreviations to games
short_names_map = {
    "Grand Theft Auto (1997)":                            "GTA (1997)",
    "Grand Theft Auto: London 1969 (1999)":               "GTA London 1969 (1999)",
    "Grand Theft Auto: London 1961 (1999)":               "GTA London 1961 (1999)",
    "Grand Theft Auto 2 (1999)":                          "GTA 2 (1999)",
    "Grand Theft Auto III (2001)":                        "GTA III (2001)",
    "Grand Theft Auto: Vice City (2002)":                 "GTA VC (2002)",
    "Grand Theft Auto: San Andreas (2004)":               "GTA SA (2004)",
    "Grand Theft Auto Advance (2004)":                    "GTA Advance (2004)",
    "Grand Theft Auto: Liberty City Stories (2005)":      "GTA LCS (2005)",
    "Grand Theft Auto: Vice City Stories (2006)":         "GTA VCS (2006)",
    "Grand Theft Auto IV (2008)":                         "GTA IV (2008)",
    "Grand Theft Auto IV: The Lost and Damned (2009)":    "GTA IV TLAD (2009)",
    "Grand Theft Auto IV: The Ballad of Gay Tony (2009)": "GTA IV TBoGT (2009)",
    "Grand Theft Auto: Chinatown Wars (2009)":            "GTA CTW (2009)",
    "Grand Theft Auto V and Online (2013 ff.)":           "GTA V (2013)"
}

# Map titles to folders
folder_mapping = {
    "gta":         "Grand Theft Auto (1997)",
    "gta 2":       "Grand Theft Auto 2 (1999)",
    "gta iii":     "Grand Theft Auto III (2001)",
    "gta vc":      "Grand Theft Auto: Vice City (2002)",
    "gta sa":      "Grand Theft Auto: San Andreas (2004)",
    "gta advance": "Grand Theft Auto Advance (2004)",
    "gta lcs":     "Grand Theft Auto: Liberty City Stories (2005)",
    "gta vcs":     "Grand Theft Auto: Vice City Stories (2006)",
    "gta iv":      "Grand Theft Auto IV (2008)",
    "gta ctw":     "Grand Theft Auto: Chinatown Wars (2009)",
    "gta v":       "Grand Theft Auto V and Online (2013 ff.)"
}

# Test once whether mathtext parser of installed matplotlib knows `\mathbfit`
try:
    matplotlib.mathtext.MathTextParser('path').parse(r'$\mathbfit{x}$')
    supports_bold_italic = True
except Exception:
    supports_bold_italic = False

# Define function to italicize game titles in plot labels
def italicize_game_label(label, bold = False):
    """
    Wraps the title of a game in mathtext italics while leaving the release year upright.
    Applies to plot labels only.
    """
    
    # Split off the release year at the last opening bracket
    title, separator, year = label.rpartition(" (")
    
    # Fall back to the whole label if it carries no year
    if not separator:
        title, year = label, None
    
    # Select font command and fall back to regular italics wherever bold italics are unavailable
    command = r'\mathbfit' if bold and supports_bold_italic else r'\mathit'
    
    # Split title at colons
    pieces = []
    
    for segment in re.split(r'(:\s*)', title):
        
        # Skip empty segments
        if not segment:
            continue
        
        # Set punctuation upright and outside the italic group, since mathtext would otherwise space it as an operator
        if segment.startswith(':'):
            pieces.append(r'\!\mathrm{:}\,')
        
        # Wrap the remaining text, where spaces need an explicit escape
        else:
            pieces.append(command + '{' + segment.replace(' ', r'\ ') + '}')
    
    # Assemble math group
    italic_title = '$' + ''.join(pieces) + '$'
    
    # Reattach year
    return italic_title if year is None else f"{italic_title} ({year}"

# Define function to normalize folder names
def normalize_folder_name(part):
    """
    Normalizes a folder name so that separators do not break the mapping.
    """
    
    # Replace underscores, hyphens, and dots by spaces; collapse repeated spaces; strip the result
    return re.sub(r'\s+', ' ', re.sub(r'[_\-.]+', ' ', part.lower())).strip()

# Define function to identify game titles
def get_game_label(path_string):
    """
    Identifies game title from file path.
    Handles DLC separation for GTA IV.
    Maps titles to folder names.
    """
    
    # Normalize paths
    p = str(path_string).lower().replace('\\', '/')
    
    # Check for DLCs
    if '1969' in p:
        return "Grand Theft Auto: London 1969 (1999)"
    if '1961' in p:
        return "Grand Theft Auto: London 1961 (1999)"
    if 'tlad' in p:
        return "Grand Theft Auto IV: The Lost and Damned (2009)"
    if 'tbogt' in p:
        return "Grand Theft Auto IV: The Ballad of Gay Tony (2009)"
    
    # Extract folder names
    parts = p.split('/')
    
    # Return first path segment that matches the mapping
    for part in parts:
        key = normalize_folder_name(part)
        if key in folder_mapping:
            return folder_mapping[key]
    
    # Fall back to »Unknown«
    return "Unknown"


############
# CLEANING #
############

# Match structural lines of the GXT dumps
section_pattern     = re.compile(r'^\[[^\]]*\]$')
header_pattern      = re.compile(r'^(Version|CharSize|NeedDecode|SingleFileTable)\b|^[A-Z]{4},\s*\d+$')

# Match format of the TXT dumps
key_comment_pattern = re.compile(r'^\S+[ \t]+//[ \t]?(.*)$')

# Match format of the deciphered GTA London files
bracket_key_pattern = re.compile(r'^\[[^\]]{1,24}\](.*)$')

# Match debug strings of the GTA Liberty City Stories dump
debug_pattern       = re.compile(r'^(?:ROW|ID)\s*:\s*(?:\d+|-+)|^MODELL?\s*:')

# Define function to filter technical artefacts
def is_technical_string(text):
    """
    Filters technical artefacts.
    """
    
    # Discard identifiers using underscores
    if '_' in text: return True
    
    # Discard lines without a single letter
    if re.fullmatch(r'[\d\W_]+', text): return True
    
    # Discard level editor artefacts
    if debug_pattern.match(text): return True
    
    # Discard single upper-case words containing a digit
    if len(text) > 1 and text.isupper() and ' ' not in text and re.search(r'\d', text): return True
    
    # Keep everything else
    return False

# Define function to extract game text
def extract_game_text(unit, ext):
    """
    Extracts and cleans a single text unit based on file extension.
    Expects one line for OXT and TXT files and one parsed row for CSV files.
    """
    
    # Process parsed CSV rows
    if ext == '.csv':
        fields = [str(f).strip() for f in unit]
        if len(fields) < 2: return None
        if fields[0].lower() in ['gxt', 'id', 'key']: return None
        text = fields[1]
    
    # Process lines
    else:
        line = str(unit).strip()
        
        # Skip structural lines
        if not line or line in ['{', '}'] or header_pattern.match(line):
            return None
        
        text = None
        
        # Split format of the OXT dumps
        if ext == '.oxt':
            if '=' in line: text = line.split('=', 1)[1].strip()
        
        # Handle TXT formats
        elif ext == '.txt':
            
            # Skip table markers
            if section_pattern.match(line): return None
            
            # Split key and text at the tabulator
            match = key_comment_pattern.match(line)
            
            if match:
                text = match.group(1)
            else:
                
                # Strip bracketed key of the GTA London files; falls back to untouched line if no key is present
                match = bracket_key_pattern.match(line)
                text  = match.group(1) if match else line
    
    # Clean extracted text
    if text:
        
        # Remove markup
        text = re.sub(r'~.*?~|#|<<|>>', '', text)
        
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Remove technical strings
        if text and not is_technical_string(text):
            return text if len(text) > 1 else None
    
    # Signal that unit holds no usable text
    return None


#############
# WILDCARDS #
#############

# Define function to convert a single search pattern
def wildcard_to_regex(wildcard_pattern):
    """
    Converts search patterns into regex objects.
    """

    # Escape the pattern; turn escaped asterisk back into a wildcard; anchor both ends at word boundaries
    regex_str = r'\b' + re.escape(wildcard_pattern).replace(r'\*', r'\w*') + r'\b'
    return re.compile(regex_str, re.IGNORECASE)

# Define function to combine the patterns of a category
def category_to_regex(patterns):
    """
    Combines all patterns of a category into a single regex object.
    """

    # Join individual patterns by alternation
    combined_pattern = "|".join([wildcard_to_regex(p).pattern for p in patterns])
    return re.compile(combined_pattern, re.IGNORECASE)

# Decipher Files for _GTA London 1969_ and _GTA London 1961_

In [ ]:
############
# FUNCTION #
############

# Define function to decipher the GTA London files
def decipher_file(input_path, output_path):
    """
    Deciphers FXT files using -1 ASCII shift.
    Restructures text by forcing new lines before every opening bracket.
    """
    
    # Convert arguments to paths
    input_file  = Path(input_path)
    output_file = Path(output_path)
    
    # Check existence of file
    if not input_file.exists():
        print("File not found.")
        return
    
    try:
        
        # Load data
        with open(input_file, 'rb') as f:
            raw_bytes = f.read()
        
        # Initialize container
        deciphered_chars = []
        
        # Shift every byte down by one; wrap around at zero
        for b in raw_bytes:
            shifted_byte = (b - 1) % 256
            char = chr(shifted_byte)
            
            # Force new line
            if char == '[':
                deciphered_chars.append('\n')
            
            # Append printable ASCII characters and new lines
            if 32 <= shifted_byte <= 126 or shifted_byte in [10, 13]:
                deciphered_chars.append(char)
        
        # Join characters into a single string
        final_text = "".join(deciphered_chars)
        
        # Export file
        with open(output_file, 'w', encoding = 'utf-8') as f:
            f.write(final_text)
        
        # Report result
        print(f"File saved to '{output_file}'.")
    
    # Handle exceptions
    except Exception as e:
        print(f"Error: {e}")


###########
# EXECUTE #
###########

# Execute
# enguk.fxt  -> gta_london_1969.txt
# enguke.fxt -> gta_london_1961.txt
input_path  = "PATH"
output_path = "PATH"

# decipher_file(input_path, output_path)

# Collect Source Files

In [ ]:
#############
# FUNCTIONS #
#############

# Define function to collect source files
def collect_source_files(root_dir):
    """
    Collects all source files of the corpus.
    Identifies duplicates by content.
    Normalizes line endings and byte order marks before hashing.
    """
    
    # Set root path and admissible extensions
    root_path        = Path(root_dir)
    valid_extensions = {'.oxt', '.txt', '.csv'}
    
    # Initialize containers
    files      = []
    duplicates = []
    empties    = []
    seen       = {}
    
    # Walk corpus in sorted order
    for file_path in sorted(root_path.rglob('*')):
        
        # Skip directories and unrelated file types
        if not file_path.is_file(): continue
        if file_path.suffix.lower() not in valid_extensions: continue
        
        # Skip exported files
        if file_path.name.startswith(export_prefix): continue
        
        # Create clean relative paths
        rel_path = str(file_path.relative_to(root_path)).replace('\\', '/')
        
        # Normalize content before hashing
        raw_bytes = file_path.read_bytes().replace(b'\r\n', b'\n').lstrip(b'\xef\xbb\xbf')
        
        # Report stub files
        if len(raw_bytes.strip().splitlines()) <= 1:
            empties.append(rel_path)
            continue
        
        # Scope duplicate check to one game
        key = (get_game_label(rel_path), hashlib.md5(raw_bytes).hexdigest())
        
        # Record duplicate and keep the first occurrence
        if key in seen:
            duplicates.append({'Duplicate': rel_path, 'Original': seen[key]})
            continue
        
        # Register file
        seen[key] = rel_path
        files.append(file_path)

    # Return files, duplicates, and empties
    return files, duplicates, empties

# Define function to read a source file as text
def read_source_text(file_path):
    """
    Reads a source file as text and tries several encodings.
    """
    
    # Try configured encodings in order and return the first that succeeds
    for encoding in source_encodings:
        try:
            return file_path.read_text(encoding = encoding)
        except UnicodeDecodeError:
            continue
    
    # Raise error rather than return an empty string
    raise ValueError(f"No matching encoding for '{file_path}'.")

# Define function to split a source file into units
def read_source_units(file_path, ext):
    """
    Reads a source file and returns its units.
    Returns lines for OXT and TXT files and parsed rows for CSV files.
    """
    
    # Decode the file
    content = read_source_text(file_path)
    
    # Parse CSV files with the csv module rather than splitting on commas
    if ext == '.csv':
        return list(csv.reader(io.StringIO(content)))
    
    # Split other files into lines
    return content.splitlines()

# Define function to extract the cleaned text of a source file
def extract_texts(file_path, ext):
    """
    Returns the cleaned text units of a source file and the number of verbatim repetitions dropped.
    """
    
    # Initialize containers
    texts    = []
    dropped  = 0
    previous = None
    
    # Iterate through the units of the file
    for unit in read_source_units(file_path, ext):
        
        # Drop units that repeat their immediate predecessor verbatim
        if previous is not None and unit == previous:
            dropped += 1
            continue
        
        # Remember unit for the next comparison
        previous = unit
        
        # Clean unit and keep it if any game text remains
        text = extract_game_text(unit, ext)
        if text:
            texts.append(text)
    
    return texts, dropped


###########
# EXECUTE #
###########

# Execute
source_files, duplicate_files, empty_files = collect_source_files(base_path)

# Report results
print(f"{len(source_files)} files collected.")
print(f"{len(duplicate_files)} duplicate files skipped.")
print(f"{len(empty_files)} stub files skipped.")

# Display duplicates
if duplicate_files:
    display(pd.DataFrame(duplicate_files))

# Calculate Lines, Characters, Tokens, Unique Types, and (Standardised) Type-Token Ratio for Raw and Cleaned Corpus

In [ ]:
#############
# FUNCTIONS #
#############

# Define function to calculate the standardised type-token ratio
def standardised_ttr(tokens, window = 1000):
    """
    Calculates the standardised type-token ratio over consecutive windows.
    """
    
    # Handle empty input
    if not tokens:
        return 0
    
    # Fall back to plain ratio if the corpus is smaller than one window
    if len(tokens) < window:
        return len(set(tokens)) / len(tokens)
    
    # Calculate the ratio for every full, non-overlapping window; drop the remainder
    ratios = [len(set(tokens[i:i + window])) / window
              for i in range(0, len(tokens) - window + 1, window)]
    
    # Average window ratios
    return sum(ratios) / len(ratios)

# Define function to compare the raw and the cleaned corpus
def calculate_metrics(files):
    """
    Calculates lines, characters, tokens, unique types, type-token ratio, and standardised type-token ratio for the raw and the cleaned corpus.
    """
    
    # Initialize containers
    raw_stats   = {'lines': 0, 'chars': 0, 'tokens': []}
    clean_stats = {'lines': 0, 'chars': 0, 'tokens': []}
    failed      = []
    
    # Collect data
    for file_path in files:
        ext = file_path.suffix.lower()
        
        try:
            
            # Process raw data line by line
            for line in read_source_text(file_path).splitlines():
                raw_line = line.strip()
                if not raw_line:
                    continue
                
                raw_stats['lines'] += 1
                raw_stats['chars'] += len(raw_line)
                raw_stats['tokens'].extend(re.findall(r'\b\w+\b', raw_line.lower()))
            
            # Process cleaned data
            clean_texts, _ = extract_texts(file_path, ext)
            
            for clean_line in clean_texts:
                clean_stats['lines'] += 1
                clean_stats['chars'] += len(clean_line)
                clean_stats['tokens'].extend(re.findall(r'\b\w+\b', clean_line.lower()))
        
        # Handle exceptions
        except Exception as e:
            failed.append({'Path': str(file_path), 'Error': f"{type(e).__name__}: {e}"})

    # Define helper function to calculate final metrics
    def compute_stats(s):
        """
        Calculates tokens (n), types (v), type-token ratio (ttr), and standardised type-token ratio (sttr).
        """
        n    = len(s['tokens'])
        v    = len(set(s['tokens']))
        ttr  = v / n if n > 0 else 0
        sttr = standardised_ttr(s['tokens'], sttr_window)
        return [s['lines'], s['chars'], n, v, ttr, sttr]
        
    # Calculate metrics for the raw and the cleaned corpus
    raw_vals   = compute_stats(raw_stats)
    clean_vals = compute_stats(clean_stats)
    
    # Calculate differences
    diff_vals = [r - c for r, c in zip(raw_vals, clean_vals)]
    diff_vals[4] = np.nan
    diff_vals[5] = np.nan
    
    # Construct DataFrame
    df_comp = pd.DataFrame({
        'Metric': [
            'Lines',
            'Characters',
            'Tokens',
            'Unique Types',
            'Type-Token Ratio',
            f'Standardised Type-Token Ratio ({sttr_window:,})'
        ],
        'Raw': raw_vals,
        'Cleaned': clean_vals,
        'Difference': diff_vals
    })
    
    # Report results
    print(f"{len(files)} files analyzed.")
    print(f"{len(failed)} files failed.")
    
    # Display failures
    if failed:
        display(pd.DataFrame(failed))
    
    # Return DataFrame
    return df_comp


###########
# EXECUTE #
###########

# Execute
audit_results = calculate_metrics(source_files)

# Display results
display(audit_results)

# Build Master Corpus

In [ ]:
############
# FUNCTION #
############

# Define function to build the master corpus
def build_master_corpus(files, root_dir):
    """
    Aggregates cleaned text from all files into a single DataFrame.
    """
    
    # Initialize containers
    all_data  = []
    failed    = []
    repeated  = []
    root_path = Path(root_dir)
    
    # Collect data
    for file_path in files:
        ext = file_path.suffix.lower()
        
        # Create clean relative paths
        rel_path = str(file_path.relative_to(root_path)).replace('\\', '/')
        
        # Map game titles to paths
        game_title = get_game_label(rel_path)
        
        try:
            
            # Clean file and count its verbatim repetitions
            texts, dropped = extract_texts(file_path, ext)
            
            # Record files affected by repetition
            if dropped:
                repeated.append({'Path': rel_path, 'Game': game_title, 'Dropped': dropped})
            
            # Store one row per line and keep its position within the file
            for position, text in enumerate(texts):
                all_data.append({
                    'Game': game_title,
                    'Path': rel_path,
                    'Position': position,
                    'Text': text
                })
        
        # Handle exceptions
        except Exception as e:
            failed.append({'Path': rel_path, 'Error': f"{type(e).__name__}: {e}"})
    
    # Report failures
    if failed:
        print(f"{len(failed)} files failed.")
        display(pd.DataFrame(failed))
    
    # Report repetitions
    if repeated:
        df_repeated = pd.DataFrame(repeated).sort_values('Dropped', ascending = False)
        print(f"{df_repeated['Dropped'].sum():,} verbatim repetitions dropped in {len(df_repeated)} files.")
        display(df_repeated)
    
    return pd.DataFrame(all_data)


###########
# EXECUTE #
###########

# Construct DataFrame
df_master = build_master_corpus(source_files, base_path)

# Export file
if not df_master.empty:
    corpus_file = export_path / f'{export_prefix}master_corpus.csv'
    
    # Write with byte order mark
    df_master.to_csv(corpus_file, index = False, encoding = 'utf-8-sig')
    
    # Report result
    print(f"Corpus saved to '{corpus_file}'.")
    
    # Display preview
    display(df_master.head(25))
else:
    print("No data found.")

# Audit Master Corpus

In [ ]:
#############
# FUNCTIONS #
#############

# Define function to audit the corpus
def audit_corpus(df):
    """
    Reports unmapped files, missing games, and repeated lines.
    """
    
    # Report unmapped files
    df_unknown = df[df['Game'] == "Unknown"]
    print(f"Unmapped lines: {len(df_unknown):,} ({len(df_unknown) / len(df) * 100:.2f} %)")
    
    # List paths responsible
    if not df_unknown.empty:
        display(df_unknown['Path'].value_counts().to_frame('Lines'))
    
    # Report missing games
    missing = [g for g in ordered_games if g not in set(df['Game'])]
    print(f"\nGames without any data: {len(missing)}")
    
    for g in missing:
        print(f"  {g}")
    
    # Report lines per game
    print("\nLines per game:")
    display(df['Game'].value_counts().reindex(ordered_games).to_frame('Lines'))
    
    # Report repeated lines within a game
    repeated = (df.groupby(['Game', 'Text'])
                  .size()
                  .reset_index(name = 'Count')
                  .query('Count > 1')
                  .sort_values('Count', ascending = False)
               )
    
    print(f"\nDistinct strings occurring more than once within a game: {len(repeated):,}")
    display(repeated.head(25))

# Define function to detect redundant dumps
def find_overlapping_files(df, threshold = 0.9, min_strings = 25):
    """
    Reports pairs of files within the same game whose extracted strings overlap almost completely.
    """
    
    # Initialize container
    pairs = []
    
    # Compare files within one game
    for game, group in df.groupby('Game'):
        
        # Reduce every file to the set of its distinct strings
        file_sets = {path: set(texts) for path, texts in group.groupby('Path')['Text']}
        
        # Drop files below the threshold
        file_sets = {path: strings for path, strings in file_sets.items() if len(strings) >= min_strings}
        
        # Sort paths
        paths = sorted(file_sets)
        
        # Iterate through pairs
        for i, path_a in enumerate(paths):
            for path_b in paths[i + 1:]:
                set_a, set_b = file_sets[path_a], file_sets[path_b]
                
                # Calculate share of the smaller file contained in the larger file
                overlap = len(set_a & set_b) / min(len(set_a), len(set_b))
                
                # Record pair if the overlap is equal to or exceeds the threshold
                if overlap >= threshold:
                    pairs.append({
                        'Game': game,
                        'File_A': path_a,
                        'File_B': path_b,
                        'Strings_A': len(set_a),
                        'Strings_B': len(set_b),
                        'Overlap': overlap
                    })
    
    # Return empty table
    if not pairs:
        return pd.DataFrame(columns = ['Game', 'File_A', 'File_B', 'Strings_A', 'Strings_B', 'Overlap'])
    
    # Return DataFrame
    return pd.DataFrame(pairs).sort_values('Overlap', ascending = False).reset_index(drop = True)


###########
# EXECUTE #
###########

# Execute
if 'df_master' in globals() and not df_master.empty:
    audit_corpus(df_master)
    
    # Report pairs with identical content
    df_overlap = find_overlapping_files(df_master)
    print(f"\nFile pairs with identical content within a game: {len(df_overlap)}")
    
    # Display results
    if not df_overlap.empty:
        display(df_overlap)

# Search for Keywords

In [ ]:
#################
# CONFIGURATION #
#################

# Configure search
search_patterns     = ["sub-dermal"]
export_results_file = export_path / f'{export_prefix}search_results.csv'


############
# FUNCTION #
############

# Define function to search for keywords
def run_search(df, patterns):
    """
    Scans the DataFrame for keywords and returns hits.
    """
    
    # Initialize container and compile the patterns once
    hits       = []
    regex_list = [(p, wildcard_to_regex(p)) for p in patterns]
    
    # Iterate through the corpus
    for row in df.itertuples(index = False):
        text = str(row.Text)
        
        # Test every pattern against the line
        for original_pattern, regex in regex_list:
            matches = regex.findall(text)
            
            # Record hit
            if matches:
                hits.append({
                    'Game': row.Game,
                    'Pattern': original_pattern,
                    'Match': matches[0],
                    'Hits': len(matches),
                    'Context': text,
                    'Path': row.Path
                })
    
    # Return DataFrame
    return pd.DataFrame(hits)


###########
# EXECUTE #
###########

# Execute
if 'df_master' in globals() and not df_master.empty:
    df_hits = run_search(df_master, search_patterns)
    
    if not df_hits.empty:
        
        # Export results
        df_hits.to_csv(export_results_file, index = False, encoding = 'utf-8-sig')
        
        # Report results
        print(f"{len(df_hits):,} lines with matches, {df_hits['Hits'].sum():,} matches in total.")
        print(f"Results exported to '{export_results_file}'.")
        
        # Display preview
        display(df_hits.head(25))
    else:
        print("No matches found.")
else:
    print("No data found.")

# Calculate Tokens, Unique Types, (Percentage of) Hapax Legomena, Average Line Length, and Average Word Length

In [ ]:
############
# FUNCTION #
############

# Define function to calculate linguistic density
def calculate_linguistic_density(df):
    """
    Calculates tokens, unique types, hapax legomena, percentage of hapax legomena, average line length, and average word length.
    """
    
    # Create counters
    word_counts          = Counter()
    line_word_counts     = []
    total_chars_in_words = 0
    total_tokens         = 0
    
    # Count
    for text in df['Text']:
        
        # Tokenize line in lower case
        found_words = word_pattern.findall(str(text).lower())
        
        # Update frequency table and the running totals
        word_counts.update(found_words)
        total_tokens += len(found_words)
        line_word_counts.append(len(found_words))
        
        # Accumulate characters for the average word length
        for w in found_words:
            total_chars_in_words += len(w)
    
    # Calculate hapax legomena
    hapax_list      = [word for word, count in word_counts.items() if count == 1]
    n_hapax         = len(hapax_list)
    vocabulary_size = len(word_counts)
    
    # Calculate hapax legomena as a share of the vocabulary
    percentage_hapax = (n_hapax / vocabulary_size) * 100 if vocabulary_size > 0 else 0
    
    # Calculate average line length
    avg_line_length = sum(line_word_counts) / len(line_word_counts) if line_word_counts else 0
    
    # Calculate average word length
    avg_word_chars = total_chars_in_words / total_tokens if total_tokens > 0 else 0
    
    # Build result table
    metrics = [
        ("Total Tokens",                 f"{total_tokens:,}"),
        ("Unique Types",                 f"{vocabulary_size:,}"),
        ("Hapax Legomena",               f"{n_hapax:,}"),
        ("Percentage of Hapax Legomena", f"{percentage_hapax:.3f} %"),
        ("Average Line Length",          f"{avg_line_length:.3f}"),
        ("Average Word Length",          f"{avg_word_chars:.3f}")
    ]
    
    # Return DataFrame
    return pd.DataFrame(metrics, columns = ['Metric', 'Value'])


###########
# EXECUTE #
###########

# Execute
if 'df_master' in globals():
    df_adv_metrics = calculate_linguistic_density(df_master)
    
    # Display results
    display(df_adv_metrics)
else:
    print("No data found.")

# Calculate Top Bigrams, Lexical Density, and Word Length Distribution

In [ ]:
#################
# CONFIGURATION #
#################

# Set function words
function_words = {
    "a", "about", "after", "all", "an", "and", "any", "are", "as", "at", "be",
    "been", "but", "by", "can", "did", "do", "does", "for", "from", "had", "has",
    "have", "he", "her", "him", "his", "how", "if", "in", "is", "it", "its", "me",
    "my", "no", "not", "of", "on", "or", "our", "out", "over", "she", "so", "some",
    "than", "that", "the", "their", "them", "then", "there", "these", "they",
    "this", "to", "up", "was", "we", "were", "what", "when", "which", "who",
    "will", "with", "would", "you", "your"
}


############
# FUNCTION #
############

# Define function to calculate the lexical fingerprint
def calculate_lexical_fingerprint(df):
    """
    Calculates top bigrams, lexical density, and word length distribution.
    """
    
    # Initialize containers
    all_tokens    = []
    bigram_counts = Counter()
    
    # Iterate through the corpus
    for text in df['Text']:
        
        # Apply tokenization
        line_tokens = word_pattern.findall(str(text).lower())
        all_tokens.extend(line_tokens)
        
        # Calculate bigrams
        if len(line_tokens) >= 2:
            line_bigrams = zip(line_tokens, line_tokens[1:])
            bigram_counts.update([f"{a} {b}" for a, b in line_bigrams])
    
    # Prepare metrics
    top_bigrams  = bigram_counts.most_common(25)
    word_lengths = [len(w) for w in all_tokens]
    len_dist     = Counter(word_lengths)
    
    # Calculate lexical density
    content_words_count = sum(1 for w in all_tokens if w not in function_words)
    lex_density = content_words_count / len(all_tokens) if all_tokens else 0
    
    # Return metrics
    return {
        "lexical_density":     lex_density,
        "top_bigrams":         top_bigrams,
        "length_distribution": len_dist
    }


###########
# EXECUTE #
###########

# Execute
if 'df_master' in globals():
    results = calculate_lexical_fingerprint(df_master)
    
    # Report lexical density
    print(f"Lexical density: {results['lexical_density']:.3f}")
    
    # Report top bigrams
    print("\nTop bigrams:")
    for phrase, count in results['top_bigrams']:
        print(f"\"{phrase}\": {count:,}")
    
    # Report word length distribution
    print("\nWord length distribution:")
    len_dist = results['length_distribution']
    for length in range(2, 13):
        freq = len_dist.get(length, 0)
        print(f"{length}: {freq:,}")

# Plot Word Length Distribution

In [ ]:
############
# FUNCTION #
############

# Define function to plot word length distribution
def plot_word_length_distribution(len_dist):
    """
    Creates a bar chart for the word length distribution.
    """
    
    # Set up graph
    plt.figure(figsize = (10, 5))
    
    # Filter word lengths
    plot_data = {length: len_dist.get(length, 0) for length in range(2, 13)}
    
    # Plot figure
    plt.bar(plot_data.keys(), plot_data.values(), color = '#2c7fb8', edgecolor = 'white', alpha = 0.8)
    
    # Set axis labels
    plt.xlabel('Characters per word', fontsize = 11)
    plt.ylabel('Frequency', fontsize = 11)
    
    # Limit x-axis
    plt.xticks(range(2, 13))
    plt.xlim(1.5, 12.5)
    
    # Format graph
    plt.grid(axis = 'y', linestyle = '--', alpha = 0.3)
    plt.tight_layout()
    
    # Export graph
    graph_file = export_path / f'{export_prefix}word_length_distribution.png'
    plt.savefig(graph_file, dpi = 300)
    
    # Report result
    print(f"Graph saved as '{graph_file}'.")
    plt.show()


###########
# EXECUTE #
###########

# Execute
if 'results' in globals():
    plot_word_length_distribution(results['length_distribution'])

# Count Keywords

In [ ]:
#################
# CONFIGURATION #
#################

# Set source
dataframe_name = df_master
text_column    = 'Text'


###########
# PREPARE #
###########

# Flatten dictionary to get list of all individual keywords
all_keywords = []
for category, words in conspiracy_categories.items():
    for word in words:
        all_keywords.append({'Category': category, 'Keyword': word})

# Initialize container
keyword_counts = []


###########
# EXECUTE #
###########

# Iterate through keywords and count
for entry in all_keywords:
    
    # Compile keyword and fill missing values
    pattern = wildcard_to_regex(entry['Keyword']).pattern
    series  = dataframe_name[text_column].fillna('')
    
    # Count occurrences and lines
    keyword_counts.append({
        'Category':       entry['Category'],
        'Keyword':        entry['Keyword'],
        'Total_Hits':     int(series.str.count(pattern, flags = re.IGNORECASE).sum()),
        'Lines_with_Hit': int(series.str.contains(pattern, flags = re.IGNORECASE, regex = True).sum())
    })

# Create DataFrame
counts_df = pd.DataFrame(keyword_counts)

# Sort results
counts_df = counts_df.sort_values(by = 'Total_Hits', ascending = False).reset_index(drop = True)

# Display results
print(counts_df.to_string())

# Export file
counts_file = export_path / f'{export_prefix}keyword_counts.csv'
counts_df.to_csv(counts_file, index = False, encoding = 'utf-8-sig')

# Report result
print(f"\nCounts saved to '{counts_file}'.")

# Plot Heatmap

In [ ]:
#################
# CONFIGURATION #
#################

# Set minimum number of hits required for a cell to carry colour
min_hits_heatmap = 10


#############
# FUNCTIONS #
#############

# Define function to build the cell labels
def build_heatmap_annotations(matrix_rel, matrix_abs):
    """
    Builds cell labels that pair the normalised rate with the absolute number.
    """
    
    # Copy matrix as object dtype
    labels = matrix_rel.copy().astype(object)
    
    # Iterate through cells
    for row in matrix_rel.index:
        for col in matrix_rel.columns:
            rel = matrix_rel.loc[row, col]
            
            # Leave cells without data empty
            if pd.isna(rel):
                labels.loc[row, col] = ''
            
            # Print rate above absolute number in brackets
            else:
                labels.loc[row, col] = f"{rel:.2f}\n({int(matrix_abs.loc[row, col]):,})"
    
    return labels

# Define function to plot heatmap
def plot_heatmap(df):
    """
    Creates a normalized heatmap of hits per 1,000 lines of game text.
    """
    
    # Restrict to mapped games
    df_games = df[df['Game'].isin(ordered_games)]
    
    # Calculate totals
    game_totals = df_games['Game'].value_counts()
    
    # Get categories
    category_order = list(conspiracy_categories.keys())
    
    # Initialize container
    results = []
    
    # Iterate through categories
    for cat, patterns in conspiracy_categories.items():
        
        # Combine patterns of a category into one regex
        combined_pattern = category_to_regex(patterns).pattern
        
        # Count occurrences per game
        hits = (df_games.groupby('Game')['Text']
                        .apply(lambda s: s.str.count(combined_pattern, flags = re.IGNORECASE).sum()))
        
        # Store one row per game and category
        for game, value in hits.items():
            results.append({'Game': game, 'Category': cat, 'Hits': int(value)})
    
    # Construct DataFrame
    df_hits = pd.DataFrame(results)
    
    # Fall back if nothing is found
    if df_hits.empty or df_hits['Hits'].sum() == 0:
        print("No matches found for any category.")
        return
    
    # Create matrix
    matrix_abs = df_hits.pivot(index = 'Game', columns = 'Category', values = 'Hits')
    
    # Normalize
    matrix_rel = matrix_abs.divide(game_totals, axis = 0) * 1000
    
    # Keep games without data as missing values
    matrix_abs = matrix_abs.reindex(index = ordered_games, columns = category_order)
    matrix_rel = matrix_rel.reindex(index = ordered_games, columns = category_order)
    
    # Map short names to game titles
    matrix_abs.index = [short_names_map.get(g, g) for g in matrix_abs.index]
    matrix_rel.index = [short_names_map.get(g, g) for g in matrix_rel.index]
    
    # Separate sparse cells
    labels  = build_heatmap_annotations(matrix_rel, matrix_abs)
    sparse  = matrix_abs < min_hits_heatmap
    missing = matrix_rel.isna()
    
    # Plot figure
    plt.figure(figsize = (16, 11))
    
    # Draw sparse cells in flat grey
    sns.heatmap(matrix_rel,
                annot      = labels,
                fmt        = '',
                cmap       = mcolors.ListedColormap(['#ededed']),
                mask       = (~sparse) | missing,
                cbar       = False,
                linewidths = 0.5,
                annot_kws  = {'color': '#888888', 'fontsize': 8}
               )
    
    # Draw remaining cells on the colour scale
    ax = sns.heatmap(matrix_rel,
                     annot      = labels,
                     fmt        = '',
                     cmap       = 'YlGnBu',
                     mask       = sparse | missing,
                     cbar_kws   = {'label': 'Hits per 1,000 lines of game text'},
                     linewidths = .5,
                     annot_kws  = {'fontsize': 8}
                    )
    
    # Italicize game titles
    ax.set_yticklabels([italicize_game_label(t.get_text()) for t in ax.get_yticklabels()])
    
    # Set axis labels
    plt.xlabel('Thematic Dimension', fontsize = 12, labelpad = 15)
    plt.ylabel('Game', fontsize = 12, labelpad = 15)
    plt.tight_layout()
    
    # Export graph
    graph_file = export_path / f'{export_prefix}heatmap.png'
    plt.savefig(graph_file, dpi = 300)
    
    # Report result
    print(f"Graph saved as '{graph_file}'.")


###########
# EXECUTE #
###########

# Execute
if 'df_master' in globals() and not df_master.empty:
    plot_heatmap(df_master)

# Plot Barcode

In [ ]:
#############
# FUNCTIONS #
#############

# Define function to build the token stream
def build_corpus_token_stream(df):
    """
    Builds an ordered token stream of the corpus and records game boundaries.
    """
    
    # Set order
    df_filtered               = df[df['Game'].isin(ordered_games)].copy()
    df_filtered['Sort_Order'] = df_filtered['Game'].map({g: i for i, g in enumerate(ordered_games)})
    df_sorted                 = df_filtered.sort_values('Sort_Order', kind = 'stable').reset_index(drop = True)
    
    # Initialize containers
    all_words       = []
    game_boundaries = []
    current_game    = None
    
    # Concatenate corpus into one list of tokens
    for row in df_sorted.itertuples(index = False):
        
        # Record token index at which a new game begins
        if row.Game != current_game:
            game_boundaries.append({'start': len(all_words), 'name': row.Game})
            current_game = row.Game
        
        # Append tokens of the line
        all_words.extend(sequence_pattern.findall(str(row.Text).lower()))
    
    # Map character offsets to token indices
    offsets, position = [], 0
    for token in all_words:
        offsets.append(position)
        position += len(token) + 1
    
    # Return both representations
    return all_words, game_boundaries, offsets, " ".join(all_words)

# Define function to locate pattern hits
def find_token_indices(offsets, joined, regex):
    """
    Locates pattern hits in the token stream and returns token indices.
    """

    # Map each match position back to the index of the token containing it
    return [bisect.bisect_right(offsets, m.start()) - 1 for m in regex.finditer(joined)]

# Define function to draw the background of the barcode
def draw_game_background(ax, game_boundaries, total_len):
    """
    Draws alternating backgrounds and staggered game labels.
    """
    
    # Set alternating shades and the staggered label depths
    colors      = ['#ffffff', '#f7f7f7']
    depth_steps = [-0.05, -0.1, -0.15, -0.2, -0.25, -0.3, -0.35, -0.4, -0.45, -0.5]
    
    # Iterate through games
    for i, boundary in enumerate(game_boundaries):
        
        # Determine token range of the game and its label
        start      = boundary['start']
        end        = game_boundaries[i + 1]['start'] if i + 1 < len(game_boundaries) else total_len
        short_name = italicize_game_label(short_names_map.get(boundary['name'], boundary['name']), bold = True)
        
        # Shade range and mark its right edge
        ax.axvspan(start, end, facecolor = colors[i % 2], alpha = 1.0, zorder = 0)
        ax.axvline(x = end, color = '#dddddd', lw = 0.8, zorder = 1)
        
        # Stagger labels
        if i < 9:
            drop_depth = depth_steps[i]
        else:
            drop_depth = depth_steps[-1]
        
        # Anchor label at the centre of the range
        mid = start + (end - start) / 2
        
        # Draw line from the axis to the label
        ax.annotate('',
                    xy         = (mid, 0),
                    xycoords   = ('data', 'axes fraction'),
                    xytext     = (mid, drop_depth),
                    textcoords = ('data', 'axes fraction'),
                    arrowprops = dict(arrowstyle = '-', color = '#999999', lw = 1)
                   )
        
        # Draw label
        ax.text(mid,
                drop_depth - 0.02,
                short_name,
                transform  = ax.get_xaxis_transform(),
                rotation   = 45,
                ha         = 'right',
                va         = 'top',
                fontsize   = 10,
                fontweight = 'bold' if supports_bold_italic else 'normal'
               )

# Define function to plot the barcode
def plot_barcode(df, categories):
    """
    Creates a barcode of conspiracy categories.
    """
    
    # Build token stream
    all_words, game_boundaries, offsets, joined = build_corpus_token_stream(df)
    total_len = len(all_words)
    
    # Set up graph
    category_names = list(categories.keys())
    fig, ax        = plt.subplots(figsize = (22, 10))
    
    # Draw background and labels
    draw_game_background(ax, game_boundaries, total_len)
    
    # Draw bars
    for idx, (cat_name, patterns) in enumerate(reversed(list(categories.items()))):
        
        # Combine patterns of a category into one search
        combined_indices = find_token_indices(offsets, joined, category_to_regex(patterns))
        
        # Plot hits as vertical lines
        ax.vlines(combined_indices,
                  idx - 0.35,
                  idx + 0.35,
                  colors    = '#2c7fb8',
                  alpha     = 0.5,
                  linewidth = 0.4,
                  zorder    = 2
                 )
    
    # Set style
    ax.set_yticks(range(len(category_names)))
    ax.set_yticklabels(list(reversed(category_names)), fontsize = 12, fontweight = 'bold')
    ax.set_xticks([])
    ax.set_xlim(0, total_len)
    
    # Set axis label
    ax.set_xlabel('Token position in corpus', fontsize = 13, labelpad = 160)
    
    # Export graph
    plt.subplots_adjust(bottom = 0.3)
    graph_file = export_path / f'{export_prefix}barcode_dimensions.png'
    plt.savefig(graph_file, dpi = 300, bbox_inches = 'tight')
    
    # Report result
    print(f"Graph saved as '{graph_file}'.")


###########
# EXECUTE #
###########

# Execute
if 'df_master' in globals():
    plot_barcode(df_master, conspiracy_categories)

# Plot Word Clouds

In [ ]:
#################
# CONFIGURATION #
#################

# Exclude noise
noise_filter = [
    'ain', 'allo', 'don', 'dont', 'hes', 'ill', 'll', 'oh', 're', 'san', 'uh', 've', 'youre'
]


############
# FUNCTION #
############

# Define function to plot word clouds
def plot_word_clouds(df, ordered_games_list):
    """
    Creates a grid of word clouds.
    """
    
    # Map colors to game titles
    game_colors = {
        "Grand Theft Auto (1997)":                            "bone",
        "Grand Theft Auto: London, 1969 (1999)":              "bone",
        "Grand Theft Auto: London, 1961 (1999)":              "bone",
        "Grand Theft Auto 2 (1999)":                          "PuBu",
        "Grand Theft Auto III (2001)":                        "Blues",
        "Grand Theft Auto: Vice City (2002)":                 "spring",
        "Grand Theft Auto: San Andreas (2004)":               "YlOrBr",
        "Grand Theft Auto Advance (2004)":                    "Greens",
        "Grand Theft Auto: Liberty City Stories (2005)":      "Blues",
        "Grand Theft Auto: Vice City Stories (2006)":         "spring",
        "Grand Theft Auto IV (2008)":                         "bone",
        "Grand Theft Auto IV: The Lost and Damned (2009)":    "gist_gray",
        "Grand Theft Auto IV: The Ballad of Gay Tony (2009)": "RdPu",
        "Grand Theft Auto: Chinatown Wars (2009)":            "Reds",
        "Grand Theft Auto V and Online (2013 ff.)":           "YlGnBu"
    }
    
    # Set stopwords
    custom_stopwords = set(STOPWORDS).union(set(noise_filter))
    
    # Filter for words with at least three characters
    cloud_pattern = re.compile(r'\b[a-z]{3,}\b')
    
    # Calculate grid dimensions
    num_games = len(ordered_games_list)
    cols      = 3
    rows      = math.ceil(num_games / cols)
    fig, axes = plt.subplots(rows, cols, figsize = (20, 5 * rows))
    
    # Flatten axes
    axes = np.atleast_1d(axes).flatten()
    
    # Iterate through games
    for i, game in enumerate(ordered_games_list):
        game_data = df[df['Game'] == game]
        
        # Leave panel blank if the game has no data
        if game_data.empty:
            axes[i].axis('off')
            continue
        
        # Extract text
        all_text      = " ".join(game_data['Text'].astype(str)).lower()
        words         = cloud_pattern.findall(all_text)
        filtered_text = " ".join([w for w in words if w not in custom_stopwords])
        
        if filtered_text:
            
            # Select color
            color_name     = game_colors.get(game, 'viridis')
            base_cmap      = plt.get_cmap(color_name)
            truncated_cmap = mcolors.ListedColormap(base_cmap(np.linspace(0.4, 0.9, 256)))
            
            # Generate cloud
            wc = WordCloud(background_color = 'white',
                           colormap         = truncated_cmap,
                           max_words        = 15,
                           width            = 800,
                           height           = 500,
                           collocations     = False,
                           random_state     = 42
                          ).generate(filtered_text)
            
            # Draw cloud
            axes[i].imshow(wc, interpolation = 'bilinear')
            
            # Set titles
            display_name = italicize_game_label(short_names_map.get(game, game), bold = True)
            axes[i].set_xlabel(display_name,
                               fontsize   = 18,
                               fontweight = 'bold' if supports_bold_italic else 'normal',
                               labelpad   = 12)
        
        # Hide axis ticks
        axes[i].set_xticks([])
        axes[i].set_yticks([])
        for spine in axes[i].spines.values():
            spine.set_visible(False)
    
    # Remove unused subplots
    for j in range(num_games, len(axes)):
        fig.delaxes(axes[j])
    
    # Export graph
    plt.tight_layout(pad = 4.0)
    graph_file = export_path / f'{export_prefix}word_clouds.png'
    plt.savefig(graph_file, dpi = 300, bbox_inches = 'tight')
    
    # Report result
    print(f"Graph saved as '{graph_file}'.")
    plt.show()


###########
# EXECUTE #
###########

# Execute
if 'df_master' in globals() and 'ordered_games' in globals():
    plot_word_clouds(df_master, ordered_games)